In [1]:
from mistralai import Mistral
import os
import pandas as pd 
import faiss
import numpy as np
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter
import time


In [2]:
# Initialiser l'encodeur (utilisez le modèle approprié)
encoding = tiktoken.get_encoding("cl100k_base")  # Encodeur standard pour la plupart des modèles

# Calculer les tokens pour une description
def count_tokens(text):
    return len(encoding.encode(text))

In [3]:
# Load file
source_data = pd.read_json('../data/evenements-publics-openagenda_26.json')

print(f'Nombre de lignes avant filtrage {source_data.shape[0]}')

# Filtrer les lignes de source_data avec des descriptions vide ou avec uniquement des espaces
source_data = source_data[source_data["longdescription_fr"].apply(lambda x: isinstance(x, str) and bool(x.strip()))]

print(f'Nombre de lignes après filtrage {source_data.shape[0]}')

long_desc_list = source_data["longdescription_fr"].tolist()

print(f'Nombre de lignes dans long_desc_list {len(long_desc_list)}')


Nombre de lignes avant filtrage 7111
Nombre de lignes après filtrage 6544
Nombre de lignes dans long_desc_list 6544


In [4]:
# Filtrer événements
# Convert both dates to UTC timezone for comparison
current_date = pd.Timestamp.now(tz='UTC')
print(f"Date actuelle de référence : {current_date}")

# Convertir les colonnes de dates en datetime AVANT le filtrage
source_data["firstdate_begin"] = pd.to_datetime(source_data["firstdate_begin"], utc=True)
source_data["firstdate_end"] = pd.to_datetime(source_data["firstdate_end"], utc=True)
source_data["lastdate_begin"] = pd.to_datetime(source_data["lastdate_begin"], utc=True)
source_data["lastdate_end"] = pd.to_datetime(source_data["lastdate_end"], utc=True)

# Filtrer les événements dont la date de FIN est postérieure à aujourd'hui
source_data = source_data[
    source_data["lastdate_end"] >= current_date
]

print(f'Nombre de lignes après filtrage des événements futurs : {source_data.shape[0]}')

# Vérification : afficher quelques dates pour contrôle
print("\n=== Vérification des dates ===")
print(source_data[["uid", "firstdate_begin", "lastdate_end"]].head(10))

Date actuelle de référence : 2026-05-19 09:57:00.568890+00:00
Nombre de lignes après filtrage des événements futurs : 2706

=== Vérification des dates ===
         uid           firstdate_begin              lastdate_end
5   70031148 2026-06-18 07:00:00+00:00 2026-06-18 08:00:00+00:00
6   56887708 2026-05-31 18:00:00+00:00 2026-05-31 19:00:00+00:00
7   31492252 2026-07-03 18:00:00+00:00 2026-07-03 20:30:00+00:00
8   40980456 2026-05-20 16:30:00+00:00 2026-05-20 18:00:00+00:00
9   90146175 2026-06-13 12:30:00+00:00 2026-06-13 14:00:00+00:00
11  69844964 2026-06-05 08:00:00+00:00 2026-06-07 16:00:00+00:00
12  99876723 2026-06-06 06:00:00+00:00 2026-06-06 10:00:00+00:00
13  36187635 2026-06-06 07:00:00+00:00 2026-06-06 15:30:00+00:00
15  84771205 2026-06-03 12:00:00+00:00 2026-06-03 14:00:00+00:00
16  53900658 2026-05-23 15:00:00+00:00 2026-05-23 21:59:00+00:00


In [5]:
# Ajouter cette cellule après le filtrage pour vérifier
print("\n=== VÉRIFICATION DES DATES ===")
current_date = pd.Timestamp.now(tz='UTC')

# Vérifier s'il reste des événements passés
past_events = source_data[
    pd.to_datetime(source_data["lastdate_end"], utc=True) < current_date
]

if len(past_events) > 0:
    print(f"⚠️ ATTENTION : {len(past_events)} événements passés détectés !")
    print(past_events[["uid", "firstdate_begin", "lastdate_end"]].head())
else:
    print(f"✓ Tous les {len(source_data)} événements sont futurs")

# Statistiques temporelles
print(f"\nDate la plus proche : {source_data['firstdate_begin'].min()}")
print(f"Date la plus lointaine : {source_data['lastdate_end'].max()}")



=== VÉRIFICATION DES DATES ===
✓ Tous les 2706 événements sont futurs

Date la plus proche : 2026-01-10 09:00:00+00:00
Date la plus lointaine : 2028-07-28 16:00:00+00:00


In [6]:
# List all event on july 2026
july_2026_events = source_data[source_data["firstdate_begin"].dt.month == 7]
july_2026_events = july_2026_events[july_2026_events["firstdate_begin"].dt.year == 2026]
print(f"\nÉvénements de juillet 2026 : {len(july_2026_events)}")
print(july_2026_events[["uid", "firstdate_begin", "lastdate_end"]])



Événements de juillet 2026 : 173
           uid           firstdate_begin              lastdate_end
7     31492252 2026-07-03 18:00:00+00:00 2026-07-03 20:30:00+00:00
61     1507761 2026-07-23 12:00:00+00:00 2026-08-06 12:30:00+00:00
63    12189212 2026-07-24 18:30:00+00:00 2026-07-24 21:00:00+00:00
200   25918870 2026-07-30 11:45:00+00:00 2026-07-30 13:30:00+00:00
319   69062381 2026-07-04 16:00:00+00:00 2026-07-04 21:00:00+00:00
...        ...                       ...                       ...
6871  42958907 2026-07-23 09:00:00+00:00 2026-07-23 10:00:00+00:00
6964  69972996 2026-07-16 11:30:00+00:00 2026-07-16 13:00:00+00:00
7068  19169273 2026-07-24 18:30:00+00:00 2026-07-24 20:30:00+00:00
7072  18973857 2026-07-03 17:00:00+00:00 2026-07-03 21:00:00+00:00
7075  48812994 2026-07-05 06:00:00+00:00 2026-11-22 16:00:00+00:00

[173 rows x 3 columns]


In [7]:
# Analyser vos descriptions
source_data["token_count"] = source_data["longdescription_fr"].apply(count_tokens)

# Statistiques
print(f"Nombre moyen de tokens: {source_data['token_count'].mean():.2f}")
print(f"Nombre max de tokens: {source_data['token_count'].max()}")
print(f"Nombre min de tokens: {source_data['token_count'].min()}")

# Afficher la distribution
print("\nDistribution des tokens:")
print(source_data["token_count"].describe())

# Voir les descriptions les plus longues
print("\nTop 5 descriptions les plus longues:")
print(source_data.nlargest(5, "token_count")[["uid", "longdescription_fr", "token_count"]])

Nombre moyen de tokens: 168.71
Nombre max de tokens: 3155
Nombre min de tokens: 11

Distribution des tokens:
count    2706.000000
mean      168.708056
std       179.153742
min        11.000000
25%        97.000000
50%       133.000000
75%       157.000000
max      3155.000000
Name: token_count, dtype: float64

Top 5 descriptions les plus longues:
           uid                                 longdescription_fr  token_count
5450  39941737  <p>𝗟𝗲 Museon Arlaten , musée de Provence et le...         3155
4032  90961540  <h2>Valérie Jouve, (en sa présence) : <em>Les ...         2478
5598  15644546  <h2>Performance de Mouna Ahizoune, <em>When i ...         2409
7023  12339413  <p>Enfermée dans un camp du simple fait de ses...         1977
3029  24200773  <p><em>Une invitation de Lieux Publics à inves...         1677


In [8]:
print("=== COLONNES DISPONIBLES DANS source_data ===")
print(source_data.columns.tolist())

print("\n=== EXEMPLE DE DONNÉES GÉOGRAPHIQUES ===")
print(source_data[['uid', 'location_name', 'location_city', 'location_postalcode']].head(10))

# Vérifier les valeurs uniques
print(f"\n📍 Nombre de villes uniques : {source_data['location_city'].nunique()}")
print(f"📮 Nombre de codes postaux uniques : {source_data['location_postalcode'].nunique()}")

=== COLONNES DISPONIBLES DANS source_data ===
['uid', 'slug', 'canonicalurl', 'title_fr', 'description_fr', 'longdescription_fr', 'conditions_fr', 'keywords_fr', 'image', 'imagecredits', 'thumbnail', 'originalimage', 'updatedat', 'daterange_fr', 'firstdate_begin', 'firstdate_end', 'lastdate_begin', 'lastdate_end', 'timings', 'accessibility', 'accessibility_label_fr', 'location_uid', 'location_coordinates', 'location_name', 'location_address', 'location_district', 'location_insee', 'location_postalcode', 'location_city', 'location_department', 'location_region', 'location_countrycode', 'location_image', 'location_imagecredits', 'location_phone', 'location_website', 'location_links', 'location_tags', 'location_description_fr', 'location_access_fr', 'attendancemode', 'onlineaccesslink', 'status', 'age_min', 'age_max', 'originagenda_title', 'originagenda_uid', 'contributor_email', 'contributor_contactnumber', 'contributor_contactname', 'contributor_contactposition', 'contributor_organizati

## Chunking

Après analyse des données, le constat est le suivant : 
- 75% des descriptions < 153 tokens → Pas besoin de chunking
- Max 1983 tokens → Quelques descriptions très longues nécessitent un découpage
- Moyenne 151 tokens → Largement sous la limite de 512 tokens

Conclusion, nous allons utiliser le chunking conditionnel en découpant uniquement les descriptions qui sont supérieures à 512 tokens.

In [9]:
# Vérifier la disponibilité des informations de contact
print("\n=== ANALYSE DES DONNÉES DE CONTACT ===")

contact_columns = ['location_phone', 'location_website']
for col in contact_columns:
    if col in source_data.columns:
        non_null = source_data[col].notna().sum()
        percentage = (non_null / len(source_data)) * 100
        print(f"✓ {col}: {non_null}/{len(source_data)} ({percentage:.1f}%)")
        
        # Afficher quelques exemples
        if non_null > 0:
            print(f"  Exemples: {source_data[col].dropna().head(3).tolist()}")
    else:
        print(f"✗ {col}: Colonne absente")

print("\n=== STATISTIQUES DE LOCALISATION ===")
location_columns = ['location_name', 'location_address', 'location_city', 'location_postalcode']
for col in location_columns:
    if col in source_data.columns:
        non_null = source_data[col].notna().sum()
        percentage = (non_null / len(source_data)) * 100
        print(f"✓ {col}: {non_null}/{len(source_data)} ({percentage:.1f}%)")



=== ANALYSE DES DONNÉES DE CONTACT ===
✓ location_phone: 772/2706 (28.5%)
  Exemples: ['0491145880', '0486971988', '04 90 69 75 66']
✓ location_website: 830/2706 (30.7%)
  Exemples: ['https://vieille-charite-marseille.com', 'https://lafabulerie.com', 'http://www.plein-pagnier.com']

=== STATISTIQUES DE LOCALISATION ===
✓ location_name: 2706/2706 (100.0%)
✓ location_address: 2706/2706 (100.0%)
✓ location_city: 2705/2706 (100.0%)
✓ location_postalcode: 2697/2706 (99.7%)


In [10]:
# Configuration du splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,  # Taille cible en tokens (marge de sécurité)
    chunk_overlap=50,  # Chevauchement pour garder le contexte
    length_function=count_tokens,
    separators=["\n\n", "\n", ". ", " ", ""]  # Découpe intelligente
)

def process_description(row):
    """Découpe uniquement si nécessaire ET PRÉSERVE LES MÉTADONNÉES"""
    text = row["longdescription_fr"]
    token_count = row["token_count"]
    uid = row["uid"]
    
    # ✅ AJOUTER TOUTES LES MÉTADONNÉES NÉCESSAIRES
    firstdate_begin = row["firstdate_begin"]
    firstdate_end = row["firstdate_end"]
    lastdate_begin = row["lastdate_begin"]
    lastdate_end = row["lastdate_end"]
    
    # ✅ NOUVELLES MÉTADONNÉES GÉOGRAPHIQUES
    location_name = row.get("location_name", None)
    location_city = row.get("location_city", None)
    location_postalcode = row.get("location_postalcode", None)
    location_phone = row.get("location_phone", None)
    location_website = row.get("location_website", None)
    
    if token_count <= 512:
        # Pas de chunking nécessaire
        return [{
            "uid": uid,
            "chunk_id": 0,
            "text": text,
            "token_count": token_count,
            "firstdate_begin": firstdate_begin,
            "firstdate_end": firstdate_end,
            "lastdate_begin": lastdate_begin,
            "lastdate_end": lastdate_end,
            # ✅ AJOUTER LES MÉTADONNÉES GÉOGRAPHIQUES
            "location_name": location_name,
            "location_city": location_city,
            "location_postalcode": location_postalcode,
            "location_phone": location_phone,
            "location_website": location_website
        }]
    else:
        # Chunking pour les longues descriptions
        chunks = text_splitter.split_text(text)
        return [{
            "uid": uid,
            "chunk_id": i,
            "text": chunk,
            "token_count": count_tokens(chunk),
            "firstdate_begin": firstdate_begin,
            "firstdate_end": firstdate_end,
            "lastdate_begin": lastdate_begin,
            "lastdate_end": lastdate_end,
            # ✅ RÉPÉTER POUR CHAQUE CHUNK
            "location_name": location_name,
            "location_city": location_city,
            "location_postalcode": location_postalcode,
            "location_phone": location_phone,
            "location_website": location_website
        } for i, chunk in enumerate(chunks)]

# Appliquer le traitement
chunks_list = []
for _, row in source_data.iterrows():
    chunks_list.extend(process_description(row))

# Créer un nouveau DataFrame avec les chunks
chunks_df = pd.DataFrame(chunks_list)

print(f"Nombre de descriptions originales: {len(source_data)}")
print(f"Nombre total de chunks: {len(chunks_df)}")
print(f"Descriptions découpées: {len(chunks_df[chunks_df['chunk_id'] > 0]['uid'].unique())}")

Nombre de descriptions originales: 2706
Nombre total de chunks: 2929
Descriptions découpées: 107


## Embedding
Utilisation de Mistral pour l'embedding

In [11]:
def create_smart_batches(texts, max_tokens_per_batch=8000):
    """Crée des batches en fonction du nombre de tokens"""
    batches = []
    current_batch = []
    current_tokens = 0
    
    for text in texts:
        text_tokens = count_tokens(text)
        
        if current_tokens + text_tokens > max_tokens_per_batch and current_batch:
            batches.append(current_batch)
            current_batch = [text]
            current_tokens = text_tokens
        else:
            current_batch.append(text)
            current_tokens += text_tokens
    
    if current_batch:
        batches.append(current_batch)
    
    return batches

In [13]:
# ✅ ÉTAPE 1 : Créer la colonne enrichie
def create_enriched_text(row):
    """Texte enrichi pour embedding de meilleure qualité"""
    parts = [row['text']]
    
    # Ajouter localisation si disponible
    if pd.notna(row.get('location_city')):
        parts.append(f"Ville: {row['location_city']}")
    
    if pd.notna(row.get('location_name')):
        parts.append(f"Lieu: {row['location_name']}")
    
    # Ajouter date si disponible
    if pd.notna(row.get('firstdate_begin')):
        try:
            date_obj = pd.to_datetime(row['firstdate_begin'])
            # Traduire le mois en français
            mois_fr = {
                1: 'janvier', 2: 'février', 3: 'mars', 4: 'avril',
                5: 'mai', 6: 'juin', 7: 'juillet', 8: 'août',
                9: 'septembre', 10: 'octobre', 11: 'novembre', 12: 'décembre'
            }
            date_str = f"{mois_fr[date_obj.month]} {date_obj.year}"
            parts.append(f"Date: {date_str}")
        except:
            pass
    
    return " | ".join(parts)

chunks_df['text_for_embedding'] = chunks_df.apply(create_enriched_text, axis=1)

print("\n=== VÉRIFICATION ===")
print(f"✓ Colonne créée : {len(chunks_df['text_for_embedding'])} entrées")
print(f"\nExemple :")
print(chunks_df[['text', 'text_for_embedding']].iloc[0])

# ✅ ÉTAPE 2 : Créer les batches
texts_to_embed = chunks_df["text_for_embedding"].tolist()
batches = create_smart_batches(texts_to_embed, max_tokens_per_batch=7000)
print(f"\n✓ {len(batches)} batches créés")


=== VÉRIFICATION ===
✓ Colonne créée : 2929 entrées

Exemple :
text                  <p>Venez vous informez sur les opportunités d'...
text_for_embedding    <p>Venez vous informez sur les opportunités d'...
Name: 0, dtype: str

✓ 78 batches créés


In [ ]:
batches = create_smart_batches(texts_to_embed, max_tokens_per_batch=7000)
print(f"Nombre de batches créés: {len(batches)}")

all_embeddings = []

with Mistral(
    api_key=os.getenv("MISTRAL_API_KEY", ""),
) as mistral:
    
    for i, batch in enumerate(batches):
        print(f"Batch {i+1}/{len(batches)} - {len(batch)} textes")
        
        try:
            res = mistral.embeddings.create(
                model="mistral-embed", 
                inputs=batch
            )
            
            all_embeddings.extend([item.embedding for item in res.data])
            print(f"Total embeddings accumulés: {len(all_embeddings)}")
            
            # Pause entre les batches pour respecter le rate limit
            if i < len(batches) - 1:  # Pas de pause après le dernier batch
                wait_time = 2  # Secondes entre chaque batch
                print(f"Pause de {wait_time}s...")
                time.sleep(wait_time)
                
        except Exception as e:
            if "429" in str(e) or "rate_limited" in str(e).lower():
                print(f"Rate limit atteint, pause de 60s...")
                time.sleep(60)
                # Réessayer le batch
                print(f"Nouvelle tentative pour le batch {i+1}")
                res = mistral.embeddings.create(
                    model="mistral-embed", 
                    inputs=batch
                )
                all_embeddings.extend([item.embedding for item in res.data])
                print(f"  ✓ Batch traité après retry")
            else:
                print(f"  ✗ Erreur: {e}")
                raise

# Vérification avant assignation
print(f"\nVérification:")
print(f"  - Nombre de chunks: {len(chunks_df)}")
print(f"  - Nombre d'embeddings: {len(all_embeddings)}")

# Ajouter les embeddings au DataFrame
chunks_df["embedding"] = all_embeddings

# ✅ Supprimer text_for_embedding avant sauvegarde (on n'en a plus besoin)
chunks_df_to_save = chunks_df.drop('text_for_embedding', axis=1)

# Sauvegarder
chunks_df_to_save.to_json("../data/chunks_with_embeddings.json", orient="records", force_ascii=False)

Nombre de batches créés: 78
Batch 1/78 - 36 textes
Total embeddings accumulés: 36
Pause de 2s...
Batch 2/78 - 44 textes
Total embeddings accumulés: 80
Pause de 2s...
Batch 3/78 - 38 textes
Total embeddings accumulés: 118
Pause de 2s...
Batch 4/78 - 38 textes
Total embeddings accumulés: 156
Pause de 2s...
Batch 5/78 - 44 textes
Total embeddings accumulés: 200
Pause de 2s...
Batch 6/78 - 46 textes
Total embeddings accumulés: 246
Pause de 2s...
Batch 7/78 - 35 textes
Total embeddings accumulés: 281
Pause de 2s...
Batch 8/78 - 33 textes
Total embeddings accumulés: 314
Pause de 2s...
Batch 9/78 - 36 textes
Total embeddings accumulés: 350
Pause de 2s...
Batch 10/78 - 31 textes
Total embeddings accumulés: 381
Pause de 2s...
Batch 11/78 - 38 textes
Total embeddings accumulés: 419
Pause de 2s...
Batch 12/78 - 43 textes
Total embeddings accumulés: 462
Pause de 2s...
Batch 13/78 - 26 textes
Total embeddings accumulés: 488
Pause de 2s...
Batch 14/78 - 39 textes
Total embeddings accumulés: 527
Paus

C:\Users\johan\AppData\Local\Temp\ipykernel_40400\2299769521.py:56: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  chunks_df_to_save.to_json("chunks_with_embeddings.json", orient="records", force_ascii=False)



=== VÉRIFICATION DU FICHIER GÉNÉRÉ ===
Colonnes présentes : ['uid', 'chunk_id', 'text', 'token_count', 'firstdate_begin', 'firstdate_end', 'lastdate_begin', 'lastdate_end', 'location_name', 'location_city', 'location_postalcode', 'location_phone', 'location_website', 'embedding']

Villes uniques : 1
Codes postaux uniques : 1


In [15]:
# ✅ VÉRIFICATION
print("\n=== VÉRIFICATION DU FICHIER GÉNÉRÉ ===")
test_load = pd.read_json("../data/chunks_with_embeddings.json")
print(f"Colonnes présentes : {test_load.columns.tolist()}")
print(f"\nVilles uniques : {test_load['location_city'].nunique()}")
print(f"Codes postaux uniques : {test_load['location_postalcode'].nunique()}")


=== VÉRIFICATION DU FICHIER GÉNÉRÉ ===
Colonnes présentes : ['uid', 'chunk_id', 'text', 'token_count', 'firstdate_begin', 'firstdate_end', 'lastdate_begin', 'lastdate_end', 'location_name', 'location_city', 'location_postalcode', 'location_phone', 'location_website', 'embedding']

Villes uniques : 230
Codes postaux uniques : 230


In [17]:
# Initialisation de l'index Faiss
embeddings_array = np.array([item for item in chunks_df["embedding"]])
dimension = embeddings_array.shape[1]  # Nombre de features
n_vectors = embeddings_array.shape[0]
print(f"Dimension des vecteurs: {dimension}")
print(f"Nombre de vecteurs à indexer: {n_vectors}")

# OPTIMISATION: Choix de l'index selon la taille
if n_vectors < 10000:
    # Pour petits datasets: IndexFlatL2 (exact, rapide)
    index = faiss.IndexFlatL2(dimension)
    print("Index utilisé: IndexFlatL2 (recherche exacte)")
else:
    # Pour grands datasets: IndexIVFFlat (approximatif, plus rapide)
    nlist = min(100, n_vectors // 39)  # Nombre de clusters
    quantizer = faiss.IndexFlatL2(dimension)
    index = faiss.IndexIVFFlat(quantizer, dimension, nlist)
    
    # Entraînement de l'index
    print(f"Entraînement de l'index avec {nlist} clusters...")
    index.train(embeddings_array)
    print("Index utilisé: IndexIVFFlat (recherche approximative rapide)")

# Ajout des embeddings à l'index
index.add(embeddings_array)

# VÉRIFICATION 2: Tous les vecteurs sont dans l'index
assert index.ntotal == n_vectors, f"Index incomplet! {index.ntotal}/{n_vectors}"
print(f"Vérification: {index.ntotal} vecteurs indexés sur {n_vectors}")

# Sauvegarde de l'index sur le disque
faiss.write_index(index, "../data/faiss_index.idx")

# VÉRIFICATION 3: Test de cohérence
print("\n=== Test de cohérence ===")
test_vector = embeddings_array[0:1]
distances, indices = index.search(test_vector, 1)
assert indices[0][0] == 0, "L'index ne retourne pas le bon vecteur!"
print("Test de cohérence réussi")

# Statistiques finales
print(f"\n=== Statistiques ===")
print(f"Événements originaux: {len(source_data)}")
print(f"Chunks créés: {len(chunks_df)}")
print(f"Vecteurs indexés: {index.ntotal}")
print(f"Taux de couverture: {(index.ntotal / len(source_data)) * 100:.1f}%")

Dimension des vecteurs: 1024
Nombre de vecteurs à indexer: 2929
Index utilisé: IndexFlatL2 (recherche exacte)
Vérification: 2929 vecteurs indexés sur 2929

=== Test de cohérence ===
Test de cohérence réussi

=== Statistiques ===
Événements originaux: 2706
Chunks créés: 2929
Vecteurs indexés: 2929
Taux de couverture: 108.2%


In [18]:
# VÉRIFICATION 4: Distribution des distances
print("\n=== Analyse de la qualité de l'index ===")

# Recherche de tous les vecteurs dans l'index
all_distances, all_indices = index.search(embeddings_array[:100], 1)

print(f"Distance moyenne à soi-même: {all_distances.mean():.6f}")
print(f"Distance max à soi-même: {all_distances.max():.6f}")

if all_distances.mean() > 0.001:
    print("Attention: Les vecteurs ne se retrouvent pas exactement")
else:
    print("Qualité de l'index: Excellente")

# VÉRIFICATION 5: Couverture des événements originaux
indexed_uids = set(chunks_df['uid'].unique())
original_uids = set(source_data['uid'].unique())

missing_uids = original_uids - indexed_uids
if missing_uids:
    print(f"{len(missing_uids)} événements manquants: {list(missing_uids)[:5]}")
else:
    print(f"Tous les {len(original_uids)} événements sont indexés")



=== Analyse de la qualité de l'index ===
Distance moyenne à soi-même: 0.000000
Distance max à soi-même: 0.000000
Qualité de l'index: Excellente
Tous les 2706 événements sont indexés


In [19]:
# ✅ DIAGNOSTIC : Vérifier les dates dans chunks_df
print("\n=== DIAGNOSTIC DES DATES ===")

# Charger le fichier sauvegardé
chunks_df = pd.read_json("../data/chunks_with_embeddings.json")

# Convertir les dates
chunks_df['lastdate_end_dt'] = pd.to_datetime(chunks_df['lastdate_end'], utc=True, errors='coerce')

# Date actuelle
current_date = pd.Timestamp.now(tz='UTC')
print(f"Date actuelle : {current_date}")

# Statistiques
print(f"\nNombre total de chunks : {len(chunks_df)}")
print(f"Nombre de dates valides : {chunks_df['lastdate_end_dt'].notna().sum()}")
print(f"Nombre de dates invalides : {chunks_df['lastdate_end_dt'].isna().sum()}")

# Événements futurs
future_events = chunks_df[chunks_df['lastdate_end_dt'] >= current_date]
print(f"\nÉvénements futurs : {len(future_events['uid'].unique())} événements uniques")
print(f"Chunks futurs : {len(future_events)} chunks")

# Afficher quelques exemples
print(f"\n📅 Exemples d'événements futurs :")
print(future_events[['uid', 'lastdate_end']].drop_duplicates('uid').head(10))

# Événements passés
past_events = chunks_df[chunks_df['lastdate_end_dt'] < current_date]
print(f"\n⚠️ Événements passés : {len(past_events['uid'].unique())} événements")


=== DIAGNOSTIC DES DATES ===
Date actuelle : 2026-05-19 11:54:39.353599+00:00

Nombre total de chunks : 2929
Nombre de dates valides : 2929
Nombre de dates invalides : 0

Événements futurs : 0 événements uniques
Chunks futurs : 0 chunks

📅 Exemples d'événements futurs :
Empty DataFrame
Columns: [uid, lastdate_end]
Index: []

⚠️ Événements passés : 2706 événements


In [ ]:
# Exemple de recherche
def search_events(query, k=5):
    """Recherche les k événements les plus similaires"""
    
    # 1. Vectoriser la requête
    with Mistral(api_key=os.getenv("MISTRAL_API_KEY", "")) as mistral:
        query_res = mistral.embeddings.create(
            model="mistral-embed",
            inputs=[query]
        )
        query_embedding = np.array([query_res.data[0].embedding])
    
    # 2. Rechercher dans FAISS
    distances, indices = index.search(query_embedding, k)
    
    # 3. Récupérer les résultats
    results = chunks_df.iloc[indices[0]]
    
    return results[["uid", "chunk_id", "text", "token_count"]], distances[0]

# Test
results, distances = search_events("concert de musique classique", k=5)
print("Top 5 événements similaires:")
print(results)
print(f"\nDistances: {distances}")